# 01 — Explore and select CB2 Ki measurements

This notebook rebuilds **the first three stages** of the rough-draft `01_data_exploration` notebook:

1. Load and inspect the raw ChEMBL download.
2. Select Ki measurements from direct human CB2 binding assays.
3. Check values and record quality, keeping review groups visible.

Run the cells from top to bottom. They read the latest **complete** acquisition in this repository's `data/raw` folder. No network call or extra Python package is needed. This notebook saves its selected measurements to `data/interim`; it does not change the raw files or export a curated modeling dataset yet.

The rough draft uses pandas tables. This version uses lists of records (Python dictionaries) to make the filters and counts visible. It preserves the draft's scientific selection rules and stops **before** RDKit structure checks, pKi conversion, and aggregation. A record reaching the end is a candidate for those later steps, not a finalized model row.

## Stage 1 — Load the raw download

`00_data_acquisition` creates a dated directory and writes `acquisition_manifest.json` only after its checks pass. Choose the newest directory with that file. This prevents accidentally reading a partial download. The same three JSON tables as in the rough draft are loaded: activities, assays, and documents.

In [1]:
from pathlib import Path
from collections import Counter
import json
import math

# Locates the project folder from the current working directory.
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
assert (project_root / "notebooks").is_dir(), "Run this notebook from the repository or notebooks folder."

# Finds the completed downloads and selects the newest one.
raw_root = project_root / "data" / "raw"
complete_downloads = sorted(raw_root.glob("chembl_cb2_*/acquisition_manifest.json"))
if not complete_downloads:
    raise FileNotFoundError("No complete download found. Run 00_data_acquisition first.")

manifest_path = complete_downloads[-1]
data_folder = manifest_path.parent
with manifest_path.open(encoding="utf-8") as file:
    manifest = json.load(file)

# Loads the activity, assay, and document records from their JSON files.
with (data_folder / "chembl_cb2_activity.json").open(encoding="utf-8") as file:
    activities = json.load(file)
with (data_folder / "chembl_cb2_assay.json").open(encoding="utf-8") as file:
    assays = json.load(file)
with (data_folder / "chembl_cb2_documents.json").open(encoding="utf-8") as file:
    documents = json.load(file)

# Compares the loaded record counts and target ID with the manifest.
assert len(activities) == manifest["activity_records"]
assert len(assays) == manifest["assay_records"]
assert len(documents) == manifest["document_records"]
assert manifest["target_chembl_id"] == "CHEMBL253"

# Prints the source folder and the sizes of the three loaded tables.
print("Source:", data_folder)
print("ChEMBL release:", manifest["chembl_version"])
print(f"Activities: {len(activities):,}")
print(f"Assays: {len(assays):,}")
print(f"Documents: {len(documents):,}")

Source: /home/justin/code/cb2-affinity/data/raw/chembl_cb2_20260922T165756862384Z
ChEMBL release: ChEMBL_37
Activities: 22,523
Assays: 1,551
Documents: 653


### See what was downloaded

`standard_type` names the measured endpoint, such as Ki or EC50. `standard_relation` tells us whether a number is exact (`=`) or a limit (`>` or `<`). At this point we only count them; we have not excluded anything.

In [2]:
# Prints selected fields from the first activity record.
example_fields = [
    "activity_id", "molecule_chembl_id", "canonical_smiles",
    "standard_type", "standard_relation", "standard_value",
    "standard_units", "assay_chembl_id", "document_chembl_id",
]
print("Example activity:")
for field in example_fields:
    print(f"  {field:<20} {activities[0].get(field)}")

# Counts distinct molecule IDs, measurement types, and measurement relations.
molecule_ids = {row.get("molecule_chembl_id") for row in activities if row.get("molecule_chembl_id")}
# Tallies each measurement type and relation in the activity records.
endpoint_counts = Counter(row.get("standard_type") for row in activities)
relation_counts = Counter(row.get("standard_relation") for row in activities)

print(f"\nDistinct molecule IDs: {len(molecule_ids):,}")
# Sizes the type column to fit the longest measurement name.
endpoint_width = max(16, max(len(str(name)) for name in endpoint_counts))
print("\nMeasurement types:")
print(f"  {'Type':<{endpoint_width}} | {'Records':>7}")
for endpoint, count in endpoint_counts.most_common():
    print(f"  {str(endpoint):<{endpoint_width}} | {count:>7,}")

# Prints relation counts with the same label and number alignment.
print("\nMeasurement relations:")
print(f"  {'Relation':<16} | {'Records':>7}")
for relation, count in relation_counts.most_common():
    print(f"  {str(relation):<16} | {count:>7,}")

Example activity:
  activity_id          35234
  molecule_chembl_id   CHEMBL158502
  canonical_smiles     N=C(NS(=O)(=O)c1ccc(F)cc1)N1CC(c2ccccc2)C(c2ccc(Cl)cc2)=N1
  standard_type        Ki
  standard_relation    >
  standard_value       1000.0
  standard_units       nM
  assay_chembl_id      CHEMBL659744
  document_chembl_id   CHEMBL1148395

Distinct molecule IDs: 11,399

Measurement types:
  Type                                                | Records
  Ki                                                  |   6,571
  EC50                                                |   5,503
  IC50                                                |   2,093
  Activity                                            |   1,845
  Emax                                                |   1,570
  kon                                                 |   1,549
  k_off                                               |   1,549
  Inhibition                                          |   1,091
  Efficacy                  

## Stage 2 — Choose direct human binding Ki records

First select `standard_type == "Ki"`. Then connect each activity to its assay by `assay_chembl_id`. An assay should have one metadata record, but it may support many activities. We check that no Ki activity loses its assay metadata.

The draft's three sequential assay rules are retained: binding assay type `B`, ChEMBL confidence score `9` for direct target assignment, and assay organism `Homo sapiens`. Requiring an explicitly human **assay** entry is stricter than relying only on the human target record. Counts show the effect of each rule.

In [3]:
# Selects activity records whose measurement type is Ki.
ki = [row for row in activities if row.get("standard_type") == "Ki"]
print(f"Ki activities: {len(ki):,}")
# Counts the distinct molecule IDs among the selected Ki activities.
print("Distinct Ki molecule IDs:", len({row.get("molecule_chembl_id") for row in ki if row.get("molecule_chembl_id")}))

# Stores assays by ID and checks that each ID appears only once.
assays_by_id = {}
for assay in assays:
    assay_id = assay["assay_chembl_id"]
    assert assay_id not in assays_by_id, "Repeated assay ID: review the raw download."
    assays_by_id[assay_id] = assay

# Copies each Ki record and adds fields from its matching assay.
ki_with_assays = []
for activity in ki:
    assay_id = activity.get("assay_chembl_id")
    assert assay_id in assays_by_id, f"Missing assay metadata for {assay_id}."
    assay = assays_by_id[assay_id]
    row = activity.copy()
    row["assay_organism"] = assay.get("assay_organism")
    row["confidence_score"] = assay.get("confidence_score")
    row["confidence_description"] = assay.get("confidence_description")
    row["relationship_type"] = assay.get("relationship_type")
    row["assay_description"] = assay.get("description")
    ki_with_assays.append(row)

# Confirms that adding assay information kept every Ki activity.
assert len(ki_with_assays) == len(ki), "The assay join changed the activity count."
print(f"Ki activities matched to assays: {len(ki_with_assays):,}")

Ki activities: 6,571
Distinct Ki molecule IDs: 5350
Ki activities matched to assays: 6,571


### Inspect the assay mix

The first table counts Ki measurements by assay type, organism, and confidence score. The next table counts measurements in each assay that fails at least one selection rule. It shows the eight assays with the most flagged measurements, along with each assay's share of all flagged measurements and a short description. This makes it easier to see whether a few assays account for most of the records set aside. The filters are applied in the next cell.

In [4]:
# Counts Ki records by assay type, organism, and confidence score.
assay_groups = Counter(
    (row.get("assay_type"), row.get("assay_organism"), row.get("confidence_score"))
    for row in ki_with_assays
)
# Prints the assay groups with aligned text labels and counts.
print(f"{'Assay type':<10} | {'Organism':<16} | {'Confidence':>10} | {'Measurements':>12}")
for group, count in assay_groups.most_common():
    print(f"{str(group[0]):<10} | {str(group[1]):<16} | {str(group[2]):>10} | {count:>12,}")

# Stores one row per failed assay and counts its Ki measurements.
excluded_assays = {}
excluded_assay_counts = Counter()
for row in ki_with_assays:
    meets_rules = (
        row.get("assay_type") == "B"
        and row.get("confidence_score") == 9
        and row.get("assay_organism") == "Homo sapiens"
    )
    # Adds each failed measurement to the count for its assay ID.
    if not meets_rules:
        assay_id = row["assay_chembl_id"]
        excluded_assays[assay_id] = row
        excluded_assay_counts[assay_id] += 1

# Sorts the failed assays by measurement count and keeps the largest eight.
excluded_measurements = sum(excluded_assay_counts.values())
top_assays = excluded_assay_counts.most_common(8)
top_eight_measurements = sum(count for assay_id, count in top_assays)

print(f"\nDistinct assays needing review: {len(excluded_assays):,}")
print(f"Measurements in those assays: {excluded_measurements:,}")
print(f"Top 8 assays: {top_eight_measurements:,} of {excluded_measurements:,} measurements "
      f"({top_eight_measurements / excluded_measurements:.1%})")
print(f"{'Assay ID':<14} | {'Type':<6} | {'Confidence':>10} | {'Organism':<16} | {'Measurements':>12} | {'Share':>7} | Description")
for assay_id, count in top_assays:
    row = excluded_assays[assay_id]
    print(f"{assay_id:<14} | "
          f"{str(row.get('assay_type')):<6} | "
          f"{str(row.get('confidence_score')):>10} | "
          f"{str(row.get('assay_organism')):<16} | "
          f"{count:>12,} | "
          f"{count / excluded_measurements:>6.1%} | "
          f"{str(row.get('assay_description'))[:120]}")

Assay type | Organism         | Confidence | Measurements
B          | Homo sapiens     |          9 |        5,825
B          | None             |          8 |          486
B          | Homo sapiens     |          8 |          186
A          | Homo sapiens     |          9 |           44
F          | Homo sapiens     |          9 |           24
B          | Cavia porcellus  |          8 |            5
F          | None             |          8 |            1

Distinct assays needing review: 58
Measurements in those assays: 746
Top 8 assays: 390 of 746 measurements (52.3%)
Assay ID       | Type   | Confidence | Organism         | Measurements |   Share | Description
CHEMBL3888919  | B      |          8 | Homo sapiens     |          121 |  16.2% | Binding Assay: The binding assays were performed by testing the ability of the new compounds to displace the radiolabele
CHEMBL925156   | B      |          8 | None             |           70 |   9.4% | Binding affinity to CB2 receptor
CHEMBL4

In [5]:
# Applies the three assay filters in order and reports the count after each one.
ki_binding = [row for row in ki_with_assays if row.get("assay_type") == "B"]
ki_direct = [row for row in ki_binding if row.get("confidence_score") == 9]
ki_human = [row for row in ki_direct if row.get("assay_organism") == "Homo sapiens"]

# Collects the records remaining after each assay filter.
assay_stages = [
    ("All Ki activities", ki_with_assays),
    ("Binding assays (B)", ki_binding),
    ("Direct target assignment (9)", ki_direct),
    ("Human assay organism", ki_human),
]
# Starts the audit with no measurements removed at the first stage.
previous_count = None
print(f"{'Stage':<32} | {'Measurements':>12} | {'Molecule IDs':>12} | {'Assays':>8} | {'Removed':>10}")
for name, rows in assay_stages:
    molecule_count = len({row.get("molecule_chembl_id") for row in rows if row.get("molecule_chembl_id")})
    assay_count = len({row["assay_chembl_id"] for row in rows})
    removed = 0 if previous_count is None else previous_count - len(rows)
    # Prints each stage with left-aligned names and right-aligned counts.
    print(f"{name:<32} | {len(rows):>12,} | {molecule_count:>12,} | {assay_count:>8,} | {removed:>10,}")
    previous_count = len(rows)

assert len(ki_human) <= len(ki_direct) <= len(ki_binding) <= len(ki_with_assays)

Stage                            | Measurements | Molecule IDs |   Assays |    Removed
All Ki activities                |        6,571 |        5,350 |      432 |          0
Binding assays (B)               |        6,502 |        5,295 |      422 |         69
Direct target assignment (9)     |        5,825 |        4,813 |      374 |        677
Human assay organism             |        5,825 |        4,813 |      374 |          0


## Stage 3 — Check measurement values and record quality

The assay rules identify relevant experiments, but some rows may still lack essential fields or a usable numeric Ki. The following checks match the draft: an activity ID, molecule ID, SMILES, and assay ID must be present; Ki must be numeric, finite, positive, and reported in `nM`.

Keep the original `standard_value`. Add a working `ki_nm` number only after checking the unit. Empty text is treated as missing. This stage does **not** validate SMILES chemically; that comes later.

In [6]:
# Lists text fields that will have blank strings changed to missing values.
text_fields = [
    "molecule_chembl_id", "canonical_smiles", "assay_chembl_id",
    "document_chembl_id", "standard_relation", "standard_units",
    "data_validity_comment",
]

# Copies each selected record before cleaning its text and numeric value.
ki_work = []
for original in ki_human:
    row = original.copy()
    for field in text_fields:
        value = row.get(field)
        if isinstance(value, str):
            value = value.strip()
            row[field] = value if value else None

    # Converts standard_value to a number, or sets it to None if conversion fails.
    try:
        row["ki_nm"] = float(row["standard_value"])
    except (TypeError, ValueError):
        row["ki_nm"] = None
    ki_work.append(row)

# Lists the fields included in the missing-value summary and counts missing values.
important_fields = [
    "activity_id", "molecule_chembl_id", "canonical_smiles",
    "assay_chembl_id", "document_chembl_id", "standard_relation",
    "standard_value", "standard_units", "data_validity_comment",
    "potential_duplicate",
]
missing_summary = []
for field in important_fields:
    missing = sum(row.get(field) is None for row in ki_work)
    missing_summary.append((field, missing))
missing_summary.sort(key=lambda item: item[1], reverse=True)

print(f"Missing values among {len(ki_work):,} selected Ki records:")
print(f"  {'Field':<24} | {'Missing':>7} | {'Missing rate':>12}")
for field, missing in missing_summary:
    print(f"  {field:<24} | {missing:>7,} | {missing / len(ki_work):>11.1%}")
print("Each rate uses all selected records; a record may be missing multiple fields.")

# Shows each value's count and share of records, from most common to least common.
for field in ["standard_relation", "standard_units", "data_validity_comment", "potential_duplicate"]:
    counts = Counter(row.get(field) for row in ki_work)
    value_width = max(24, max(len(str(value)) for value in counts))
    print(f"\n{field} (out of {len(ki_work):,} records):")
    print(f"  {'Value':<{value_width}} | {'Records':>7} | {'Percent':>8}")
    for value, count in counts.most_common():
        print(f"  {str(value):<{value_width}} | {count:>7,} | {count / len(ki_work):>8.2%}")

Missing values among 5,825 selected Ki records:
  Field                    | Missing | Missing rate
  data_validity_comment    |   5,820 |       99.9%
  standard_relation        |     228 |        3.9%
  standard_value           |     228 |        3.9%
  standard_units           |     226 |        3.9%
  activity_id              |       0 |        0.0%
  molecule_chembl_id       |       0 |        0.0%
  canonical_smiles         |       0 |        0.0%
  assay_chembl_id          |       0 |        0.0%
  document_chembl_id       |       0 |        0.0%
  potential_duplicate      |       0 |        0.0%
Each rate uses all selected records; a record may be missing multiple fields.

standard_relation (out of 5,825 records):
  Value                    | Records |  Percent
  =                        |   4,576 |   78.56%
  >                        |     990 |   17.00%
  None                     |     228 |    3.91%
  <                        |      16 |    0.27%
  >=                       | 

In [7]:
# Names the fields each usable Ki record must contain.
required_fields = ["activity_id", "molecule_chembl_id", "canonical_smiles", "assay_chembl_id"]
failed_checks = Counter()
ki_usable = []
basic_review = []

for row in ki_work:
    # Checks required fields, numeric Ki, positive Ki, and nM units for each record.
    essential_present = all(row.get(field) is not None for field in required_fields)
    numeric_and_finite = row["ki_nm"] is not None and math.isfinite(row["ki_nm"])
    positive_ki = numeric_and_finite and row["ki_nm"] > 0
    units_are_nm = row.get("standard_units") == "nM"

    checks = {
        "essential fields present": essential_present,
        "numeric and finite": numeric_and_finite,
        "positive Ki": positive_ki,
        "units are nM": units_are_nm,
    }
    for name, passed in checks.items():
        if not passed:
            failed_checks[name] += 1

    # Counts failed checks and separates usable records from records for review.
    if all(checks.values()):
        ki_usable.append(row)
    else:
        basic_review.append(row)

# Displays the number of records that failed each individual check.
print("Rows failing each check:")
print(f"  {'Check':<25} | {'Records':>7}")
for name in ["essential fields present", "numeric and finite", "positive Ki", "units are nM"]:
    print(f"  {name:<25} | {failed_checks[name]:>7,}")
print(f"Retained: {len(ki_usable):,}; set aside: {len(basic_review):,}")
# Confirms that each starting record was either retained or set aside.
assert len(ki_usable) + len(basic_review) == len(ki_work)

Rows failing each check:
  Check                     | Records
  essential fields present  |       0
  numeric and finite        |     228
  positive Ki               |     228
  units are nM              |     226
Retained: 5,597; set aside: 228


### Check whether the set-aside records have Ki values

Cell 7 kept the 228 rejected records in `basic_review`. This cell checks their standardized Ki value and original reported value. If both are missing, the record cannot provide a numeric Ki target from this download.

In [8]:
# Looks for a value in either Ki value field of each set-aside record.
rows_with_ki_value = []
for row in basic_review:
    has_standard_value = row.get("standard_value") not in (None, "")
    has_original_value = row.get("value") not in (None, "")
    if has_standard_value or has_original_value:
        rows_with_ki_value.append(row)

# Reports how many records have a value and how many do not.
print(f"Records set aside: {len(basic_review):,}")
print(f"With a Ki value: {len(rows_with_ki_value):,}")
print(f"Without a Ki value: {len(basic_review) - len(rows_with_ki_value):,}")

Records set aside: 228
With a Ki value: 0
Without a Ki value: 228


### Keep exact Ki measurements

A relation of `=` is an exact value. Relations such as `>` and `<` are **limits**: treating a bound as an exact training target would misstate the measurement. Keep limits and unusual or missing relations in separate review lists. No conversion or substitution is attempted here.

In [9]:
# Prepares separate lists for exact values, limits, and other relations.
ki_exact = []
ki_limits = []
relation_review = []
# Places each usable record into the list matching its relation.
for row in ki_usable:
    relation = row.get("standard_relation")
    if relation == "=":
        ki_exact.append(row)
    elif relation in {">", ">=", "<", "<="}:
        ki_limits.append(row)
    else:
        relation_review.append(row)

# Prints the size of each relation group.
print(f"Exact Ki: {len(ki_exact):,}")
print(f"Limit values: {len(ki_limits):,}")
print(f"Other or missing relations: {len(relation_review):,}")
# Confirms that no usable record was lost while sorting by relation.
assert len(ki_exact) + len(ki_limits) + len(relation_review) == len(ki_usable)

Exact Ki: 4,576
Limit values: 1,019
Other or missing relations: 2


### Check ChEMBL validity comments

Continue with exact Ki records whose `data_validity_comment` is blank or `Manually validated`. Keep records with other comments in `validity_review` and show those comments below.

In [10]:
# Separates exact Ki records by their validity comment.
ki_quality = []
validity_review = []
for row in ki_exact:
    if row.get("data_validity_comment") in {None, "Manually validated"}:
        ki_quality.append(row)
    else:
        validity_review.append(row)

# Shows every comment on a record set aside by this check.
print(f"Continuing after validity check: {len(ki_quality):,}")
print(f"Set aside for validity comments: {len(validity_review):,}")
if validity_review:
    print(f"{'Activity ID':>11} | Validity comment")
    for row in validity_review:
        print(f"{row['activity_id']:>11} | {row['data_validity_comment']}")

# Confirms that the two groups account for every exact Ki record.
assert len(ki_quality) + len(validity_review) == len(ki_exact)

Continuing after validity check: 4,571
Set aside for validity comments: 5
Activity ID | Validity comment
    1887239 | Potential transcription error
   16575987 | Potential transcription error
   19087977 | Potential transcription error
   19371553 | Outside typical range
   24996450 | Potential transcription error


### Review ChEMBL potential-duplicate flags

Continue with records whose `potential_duplicate` flag is `0` and keep all others in `duplicate_flag_review`. For each flagged record, the function below finds the closest value in the raw CB2 download with the same molecule, target, measurement type, relation, and unit. The 2% cutoff only groups those comparisons; it does not prove that two records came from the same experiment. [ChEMBL describes the flag here](https://chembl.gitbook.io/chembl-interface-documentation/frequently-asked-questions/chembl-data-questions).

In [11]:
# Separates records with an explicit zero flag from those needing duplicate review.
ki_unflagged = []
duplicate_flag_review = []
for row in ki_quality:
    if str(row.get("potential_duplicate")) == "0":
        ki_unflagged.append(row)
    else:
        duplicate_flag_review.append(row)
assert len(ki_unflagged) + len(duplicate_flag_review) == len(ki_quality)

# Finds the closest value among raw records with the same measurement details.
def find_closest_peer(flagged):
    closest_peer = None
    closest_gap = None
    for peer in activities:
        if peer.get("activity_id") == flagged["activity_id"]:
            continue
        if peer.get("molecule_chembl_id") != flagged.get("molecule_chembl_id"):
            continue
        if peer.get("target_chembl_id") != flagged.get("target_chembl_id"):
            continue
        if peer.get("standard_type") != flagged.get("standard_type"):
            continue
        if peer.get("standard_relation") != flagged.get("standard_relation"):
            continue
        if peer.get("standard_units") != flagged.get("standard_units"):
            continue
        try:
            peer_value = float(peer["standard_value"])
        except (TypeError, ValueError):
            continue
        if not math.isfinite(peer_value) or peer_value <= 0:
            continue
        gap = abs(peer_value - flagged["ki_nm"]) / flagged["ki_nm"] * 100
        if closest_gap is None or gap < closest_gap:
            closest_peer = peer
            closest_gap = gap
    return closest_peer, closest_gap

# Saves one closest peer and its percentage difference for each flagged record.
duplicate_review_pairs = []
for flagged in duplicate_flag_review:
    peer, gap = find_closest_peer(flagged)
    duplicate_review_pairs.append({"flagged": flagged, "peer": peer, "gap_percent": gap})

# Counts exact, close, distant, and missing value matches using a 2% review cutoff.
close_cutoff_percent = 2
exact_count = sum(pair["gap_percent"] == 0 for pair in duplicate_review_pairs)
close_count = sum(pair["gap_percent"] is not None and 0 < pair["gap_percent"] <= close_cutoff_percent
                  for pair in duplicate_review_pairs)
distant_count = sum(pair["gap_percent"] is not None and pair["gap_percent"] > close_cutoff_percent
                    for pair in duplicate_review_pairs)
distant_gaps = [pair["gap_percent"] for pair in duplicate_review_pairs
                if pair["gap_percent"] is not None and pair["gap_percent"] > close_cutoff_percent]
no_peer_count = sum(pair["gap_percent"] is None for pair in duplicate_review_pairs)
assert exact_count + close_count + distant_count + no_peer_count == len(duplicate_flag_review)

# Prints the split and the value comparisons without changing the review group.
print(f"Set aside for potential-duplicate flags: {len(duplicate_flag_review):,}")
print(f"Continuing without the flag: {len(ki_unflagged):,}")
print(f"{'Closest value':<28} | {'Flagged records':>15}")
print(f"{'Exact match':<28} | {exact_count:>15,}")
print(f"{f'Within {close_cutoff_percent}%':<28} | {close_count:>15,}")
print(f"{f'More than {close_cutoff_percent}% apart':<28} | {distant_count:>15,}")
print(f"{'No comparable record':<28} | {no_peer_count:>15,}")
if distant_gaps:
    print(f"Largest gap in the more-than-{close_cutoff_percent}% group: {max(distant_gaps):.2f}%")

Set aside for potential-duplicate flags: 490
Continuing without the flag: 4,081
Closest value                | Flagged records
Exact match                  |             399
Within 2%                    |              77
More than 2% apart           |              14
No comparable record         |               0
Largest gap in the more-than-2% group: 4.23%


### Remove identical records and check activity IDs

Two identical copies of one record should count once. JSON serialization gives us a stable comparison even when a field contains a nested list or dictionary. After removing identical copies, a repeated `activity_id` with different content would be a conflict and stop the notebook for investigation.

In [12]:
# Prepares a set of record signatures and two destination lists.
seen_records = set()
ki_records = []
duplicate_records = []
for row in ki_unflagged:
    # Sorts dictionary keys before comparing records for exact duplicates.
    signature = json.dumps(row, sort_keys=True, ensure_ascii=False)
    if signature in seen_records:
        duplicate_records.append(row)
    else:
        seen_records.add(signature)
        ki_records.append(row)

# Counts activity IDs after identical copies have been removed.
activity_id_counts = Counter(row["activity_id"] for row in ki_records)
activity_id_conflicts = [row for row in ki_records if activity_id_counts[row["activity_id"]] > 1]
# Reports duplicate copies and conflicting activity IDs.
print(f"Identical copies removed: {len(duplicate_records):,}")
print(f"Activity IDs with conflicting records: {len(set(row['activity_id'] for row in activity_id_conflicts)):,}")
assert not activity_id_conflicts, "One activity ID describes different records; investigate before continuing."
print(f"Unique activity records retained: {len(ki_records):,}")

Identical copies removed: 0
Activity IDs with conflicting records: 0
Unique activity records retained: 4,081


### Review the full selection audit

Each line is cumulative. The retained `ki_records` still represent individual **measurements**, so a molecule or structure can occur more than once. The next stage will parse molecular structures, calculate pKi, and investigate repeated measurements before choosing any one-row-per-structure modeling target.

In [13]:
# Lists the measurement groups produced at each cleaning step.
cleaning_stages = [
    ("Human binding Ki before cleaning", ki_work),
    ("Essential fields and valid nM Ki", ki_usable),
    ("Exact Ki measurements", ki_exact),
    ("Acceptable validity status", ki_quality),
    ("No potential-duplicate flag", ki_unflagged),
    ("Unique activity records", ki_records),
]
# Starts the audit before any cleaning records have been removed.
previous_count = None
print(f"{'Stage':<38} | {'Measurements':>12} | {'Removed':>10}")
for name, rows in cleaning_stages:
    removed = 0 if previous_count is None else previous_count - len(rows)
    # Prints the number retained and removed at each step.
    print(f"{name:<38} | {len(rows):>12,} | {removed:>10,}")
    previous_count = len(rows)

# Checks that the retained and review groups add up to the starting count.
assert (
    len(ki_records) + len(duplicate_records) + len(duplicate_flag_review)
    + len(validity_review) + len(ki_limits) + len(relation_review)
    + len(basic_review) == len(ki_work)
)
print("Every selected measurement is accounted for.")

Stage                                  | Measurements |    Removed
Human binding Ki before cleaning       |        5,825 |          0
Essential fields and valid nM Ki       |        5,597 |        228
Exact Ki measurements                  |        4,576 |      1,021
Acceptable validity status             |        4,571 |          5
No potential-duplicate flag            |        4,081 |        490
Unique activity records                |        4,081 |          0
Every selected measurement is accounted for.


### Choose the columns for the interim CSV

The full activity records remain available in memory and in the raw download. This cell makes a smaller row for each selected measurement and shows the seven fields that will be handed to the next notebook. The activity ID links each smaller row back to its full ChEMBL record.

In [14]:
# Names the fields needed for structure curation and source tracing.
export_columns = [
    "activity_id", "molecule_chembl_id", "canonical_smiles", "ki_nm",
    "assay_chembl_id", "document_chembl_id", "assay_description",
]

# Copies only those fields from each selected measurement.
export_rows = []
for row in ki_records:
    export_row = {field: row[field] for field in export_columns}
    export_rows.append(export_row)
assert len(export_rows) == len(ki_records)

# Shows the columns and one complete example before the CSV is written.
print(f"Measurements ready to export: {len(export_rows):,}")
print(f"Columns kept: {len(export_columns)}")
print(f"Other fields not in the interim CSV: {len(ki_records[0]) - len(export_columns)}")
print("\nColumns and example values:")
for field in export_columns:
    print(f"  {field:<20} {export_rows[0][field]}")

Measurements ready to export: 4,081
Columns kept: 7
Other fields not in the interim CSV: 45

Columns and example values:
  activity_id          353194
  molecule_chembl_id   CHEMBL63365
  canonical_smiles     COc1ccccc1CNC(=O)c1nn(CCN2CCOCC2)c2c(OC)cccc12
  ki_nm                77.0
  assay_chembl_id      CHEMBL659748
  document_chembl_id   CHEMBL1144952
  assay_description    Binding affinity against human cannabinoid receptor 2 in chinese hamster ovary cells using WIN-55212-2 mesylate[57-3H]


### Save a measurement-level checkpoint

Write the prepared `export_rows` to a CSV under `data/interim`, in a folder named for the raw download. A small JSON manifest records the source, unit, columns, and selection counts. Each row is still a measurement; structures and model targets have not been finalized.

In [15]:
import csv

# Creates an interim folder linked to the raw download used by this notebook.
interim_folder = project_root / "data" / "interim" / data_folder.name
interim_folder.mkdir(parents=True, exist_ok=True)
selected_path = interim_folder / "cb2_ki_selected.csv"
selection_manifest_path = interim_folder / "selection_manifest.json"

# Writes one CSV row per selected activity, then checks its saved activity IDs.
with selected_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=export_columns, lineterminator="\n")
    writer.writeheader()
    for row in export_rows:
        writer.writerow(row)
with selected_path.open(encoding="utf-8", newline="") as file:
    saved_rows = list(csv.DictReader(file))
assert [int(row["activity_id"]) for row in saved_rows] == [row["activity_id"] for row in export_rows]

# Records the source, selected columns, and counts after the CSV check passes.
selection_manifest = {
    "source_raw_folder": str(data_folder.relative_to(project_root)),
    "source_manifest": str(manifest_path.relative_to(project_root)),
    "chembl_version": manifest["chembl_version"],
    "selection_notebook": "notebooks/01_data_exploration.ipynb",
    "selected_file": selected_path.name,
    "selected_records": len(export_rows),
    "ki_unit": "nM",
    "columns": export_columns,
    "stage_counts": {name: len(rows) for name, rows in cleaning_stages},
    "review_counts": {
        "basic_review": len(basic_review),
        "ki_limits": len(ki_limits),
        "relation_review": len(relation_review),
        "validity_review": len(validity_review),
        "duplicate_flag_review": len(duplicate_flag_review),
        "identical_record_copies": len(duplicate_records),
    },
}
with selection_manifest_path.open("w", encoding="utf-8") as file:
    json.dump(selection_manifest, file, indent=2)
print(f"Saved {len(saved_rows):,} selected measurements to {selected_path}")
print(f"Saved selection details to {selection_manifest_path}")

Saved 4,081 selected measurements to /home/justin/code/cb2-affinity/data/interim/chembl_cb2_20260922T165756862384Z/cb2_ki_selected.csv
Saved selection details to /home/justin/code/cb2-affinity/data/interim/chembl_cb2_20260922T165756862384Z/selection_manifest.json


## Continue with structure and target curation

The selected measurement CSV is a checkpoint, not a model-ready dataset. The next stage will validate and standardize structures, calculate pKi, review conflicting measurements, and choose one target per structure. Those curated outputs belong in `data/processed`. Any feature scaling for modeling should be fitted on the training split only.